# 1. Tratamento de dados

Os passos a seguir tem por objetivo tornar os dados adequados às etapas seguintes.

In [13]:
# importa bibliotecas necessárias para tratamento de dados
import pandas as pd

In [14]:
# instancia os dados em um dataframe
df = pd.read_csv('data/planilha_usinagem.csv', sep=';')

In [15]:
# exibe informações sobre as colunas
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 185 entries, 0 to 184
Data columns (total 13 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Data                                       185 non-null    str    
 1   Fabricante do Hob                          185 non-null    str    
 2   Revestimento                               185 non-null    str    
 3   Fabricante                                 185 non-null    str    
 4   Avanço mm/rev.                             185 non-null    str    
 5   Rotação RPM                                185 non-null    int64  
 6   Vc m/min                                   185 non-null    float64
 7   Shifting mm                                185 non-null    float64
 8   Sub-Shift mm                               185 non-null    float64
 9   CT (S)                                     185 non-null    str    
 10  Quantidade de pçs por afiação Target 

Este material tem por objetivo montar uma lógica de apredizado de máquina e usa uma base dedos preenchida manualmente e obtida, e não criada, por este desenvolvedor. Por este motivo, optou-se pelo descarte dos dados faltantes. Em trabalhos futuros objetiva-se criar mecanismos de obtenção de dados mais eficazes a fim de garantir a maior robustez dos dados.

In [16]:
# exclue todas as linhas com itens faltantes
df_limpo = df.dropna()

In [17]:
# exibe informações sobre as colunas
df_limpo.info()

<class 'pandas.DataFrame'>
Index: 153 entries, 0 to 184
Data columns (total 13 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Data                                       153 non-null    str    
 1   Fabricante do Hob                          153 non-null    str    
 2   Revestimento                               153 non-null    str    
 3   Fabricante                                 153 non-null    str    
 4   Avanço mm/rev.                             153 non-null    str    
 5   Rotação RPM                                153 non-null    int64  
 6   Vc m/min                                   153 non-null    float64
 7   Shifting mm                                153 non-null    float64
 8   Sub-Shift mm                               153 non-null    float64
 9   CT (S)                                     153 non-null    str    
 10  Quantidade de pçs por afiação Target 1950 

In [18]:
# exibe os valores únicos de cada coluna

for col in df_limpo.columns: # loop para iterar sobre todas as colunas
    print(f"Valores únicos de '{col}'") # imprime o nome da coluna
    print(df_limpo[col].unique()) # imprime os valores únicos da coluna
    print('----------------------------------------------------------------------') # imprime um separador entre cada iteração

Valores únicos de 'Data'
<StringArray>
['2025-01-02 00:00:00', '2025-01-05 00:00:00', '2025-01-14 00:00:00',
 '2025-01-15 00:00:00', '2025-01-21 00:00:00', '2025-01-28 00:00:00',
 '2025-02-02 00:00:00', '2025-02-04 00:00:00', '2025-02-05 00:00:00',
 '2025-02-10 00:00:00',
 ...
 '2026-04-30 00:00:00', '2026-05-04 00:00:00', '2026-05-05 00:00:00',
 '2026-05-08 00:00:00', '2026-05-12 00:00:00', '2026-05-14 00:00:00',
 '2026-05-18 00:00:00', '2026-05-24 00:00:00', '2026-05-25 00:00:00',
 '2026-05-26 00:00:00']
Length: 126, dtype: str
----------------------------------------------------------------------
Valores únicos de 'Fabricante do Hob'
<StringArray>
['SU', 'S.U.', 'Nidec']
Length: 3, dtype: str
----------------------------------------------------------------------
Valores únicos de 'Revestimento'
<StringArray>
['Alcrona Pro', 'Alcrona Evo']
Length: 2, dtype: str
----------------------------------------------------------------------
Valores únicos de 'Fabricante'
<StringArray>
['Balzer

In [19]:
# remove sujeira da coluna 'Avanço mm/rev.'

df_limpo = df_limpo[df_limpo['Avanço mm/rev.'] != '\\'] # remove a linha com '\' da colunas 'Avanço mm/rev.'
print("Valores únicos da coluna 'Avanço mm/rev.'") # uma pequena descrição
print(df_limpo['Avanço mm/rev.'].unique()) # imprime os valores únicos da coluna

Valores únicos da coluna 'Avanço mm/rev.'
<StringArray>
['1.75', '1.4', '2.0', '1.5']
Length: 4, dtype: str


In [20]:
# corrige os valores da coluna 'CT (S)'

df_limpo['CT (S)'] = df_limpo['CT (S)'].astype(str).str.replace(',', '.') # substitue os algorítimos decimais ',' por '.' na coluna 'CT(S)'
print("Valores únicos da coluna 'CT (S)'") # uma pequena descrição
print(df_limpo['CT (S)'].unique()) # imprime os valores únicos da coluna

Valores únicos da coluna 'CT (S)'
<StringArray>
['42.9', '38.5', '46.9', '29.4']
Length: 4, dtype: str


In [21]:
df_limpo['Fabricante do Hob'].value_counts()

Fabricante do Hob
SU       124
S.U.      15
Nidec     13
Name: count, dtype: int64

Das 104 entradas remanecentes em df_limpo, apenas 1 refere-se a 'Fabricante do Hob' Nidec. Por este motivo, será considerada como coluna de valor único. Além disso, para este estudo, também será descartada a coluna data visto que o modelo não usará séries temporais

In [22]:
# elimina as colunas de valor único
df_limpo = df_limpo.drop(columns=['Data', 'Fabricante', 'Fabricante do Hob'])

In [23]:
# descarta possíveis linhas duplicadasd
df_limpo = df_limpo.drop_duplicates()

In [24]:
# converte as colunas faltantes para os tipos finais

colunas_converter_float = ['Avanço mm/rev.', 'CT (S)'] # define as colunas a serem corrigidas

for coluna in colunas_converter_float: # loop para iterar sobre as colunas a sere corrigidas
    df_corrigido = df_limpo # copia df_limpo para um novo dataframe
    df_corrigido[coluna] = df_corrigido[coluna].astype(float) # converte as colunas em colunas_converter_float para float

df_corrigido['Quantidade de pçs por afiação Target 1950'] = df_corrigido['Quantidade de pçs por afiação Target 1950'].astype(int) # converte a 'Quantidade de pçs por afiação Target 1950' para int64

df_corrigido.info() # exibe informações sobre as colunas

<class 'pandas.DataFrame'>
Index: 152 entries, 0 to 184
Data columns (total 10 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Revestimento                               152 non-null    str    
 1   Avanço mm/rev.                             152 non-null    float64
 2   Rotação RPM                                152 non-null    int64  
 3   Vc m/min                                   152 non-null    float64
 4   Shifting mm                                152 non-null    float64
 5   Sub-Shift mm                               152 non-null    float64
 6   CT (S)                                     152 non-null    float64
 7   Quantidade de pçs por afiação Target 1950  152 non-null    int64  
 8   dureza sup                                 152 non-null    float64
 9   dureza nuc                                 152 non-null    float64
dtypes: float64(7), int64(2), str(1)
memory usa

In [25]:
df_corrigido

,Revestimento,Avanço mm/rev.,Rotação RPM,Vc m/min,Shifting mm,Sub-Shift mm,CT (S),Quantidade de pçs por afiação Target 1950,dureza sup,dureza nuc
0,Alcrona Pro,1.75,350,98.960,10.0,0.666667,42.9,2620,94.3,96.5
1,Alcrona Evo,1.40,390,110.270,11.4,0.760000,38.5,2458,90.4,93.1
2,Alcrona Pro,1.75,320,90.478,11.4,0.760000,46.9,2718,98.6,94.6
3,Alcrona Evo,1.40,320,90.478,10.0,0.666667,46.9,2496,93.9,94.4
4,Alcrona Evo,2.00,510,144.199,10.0,0.666667,29.4,1872,92.0,93.2
...,...,...,...,...,...,...,...,...,...,...
177,Alcrona Evo,1.75,350,98.960,10.0,0.666667,42.9,2414,95.2,93.7
180,Alcrona Evo,1.40,510,144.199,11.4,0.760000,29.4,2336,97.5,98.2
181,Alcrona Evo,1.50,320,90.478,11.4,0.760000,46.9,2453,92.2,96.2
183,Alcrona Pro,1.40,320,90.478,10.0,0.666667,46.9,3125,94.4,92.2


# 2. Análise Exploratória dos Dados (EDA)

In [26]:
# importa bibliotecas necessárias para a EDA
import plotly.express as px

In [27]:
# cria um histograma da coluna 'Quantidade de pçs por afiação Target 1950'

fig = px.histogram( # cria uma figura para o gráfico
    df_corrigido, # define o df a ser usado
    x='Quantidade de pçs por afiação Target 1950', # define os valores de X
    nbins=10, # sugestioná o número de barras do hostograma usando o método da raiz quadrada (k = n^(1/2) ==> nbins = 104 linhas^(1/2))
    title='Distribuição da Quantidade de Peças por Afiação', # define um título para o gráfico
    labels={'Quantidade de pçs por afiação Target 1950': 'Quantidade de Peças'}, # múda o título do eixo X de 'Quantidade de pçs por afiação Target 1950' para 'Quantidade de Peças'
    color_discrete_sequence=['#1f77b4'] # define a cor padrão como azul
)

fig.update_layout(yaxis_title='Frequência') # muda o título do eixo Y de 'count' para 'Frequência'

fig.show() # exibe a figura

Através do histograma da distribuição da quantidade de peças por afiação é possível observar um comportamento normal em quase todas as faixas com excessão da faixa de 3500 a 3999 peças por afiação que apresenta uma única insidência. O que pode indicar um erro de durante a aquisição desse dado (erro de digitação, por exemplo) ou um evento anormalmente favorável à durabilidade da ferramenta.

In [28]:
# cria um gráfico de boxplot da relação entre 'Revestimento' e 'Quantidade de pçs por afiação Target 1950'

fig_box = px.box( # cria uma figura para o boxplot
    df_corrigido, # define o df a ser usado
    x='Revestimento', # define os valores de X
    y='Quantidade de pçs por afiação Target 1950', # define os valores de Y 
    color='Revestimento', # separa os dados conforme 'Revestimento'
    title='Variação da Vida Útil por Revestimento', # define um título para o gráfico
    labels={'Quantidade de pçs por afiação Target 1950': 'Peças Produzidas'} #  # múda o título do eixo X de 'Quantidade de pçs por afiação Target 1950' para 'Quantidade de Peças'
)

fig_box.show() # exibe a figura

Apesar de ser mais notável no Revestimento 'Alcrona Evo', ambos os revestimentosapresentam apresentam uma variação muito maior na vida útil do que o esperado. É necessessário investigar o motivo desta variação.

In [29]:
# cria um heatmap das variáveis numéricas de df_corrigido

df_numerico = df_corrigido.select_dtypes(include=['float64', 'int64']) # cria df_numerico com apneas as colunas numéricas de df_corrigido

matriz_correlacao = df_numerico.corr(method='spearman') # cria uma matriz de correlação das colunas de df_numerico

fig_corr = px.imshow( # cria uma figura para o heatmap
    matriz_correlacao, # define a matriz a ser usado
    text_auto=".2f", # mostra os valores com 2 casas decimais
    aspect="auto", # define a proporção do heatmap como automática forçañdo-o a preencher a altura e largura da figura
    color_continuous_scale='RdBu_r', # define a escala de cores como Azul para positivo e Vermelho para negativo
    title='Matriz de Correlação entre Parâmetros de Usinagem' # define o título do gráfico
)

fig_corr.show() # exibe a figura

Não foi possível identificar nenhuma correlação linear direta entre as features e a variável alvo 'Quantidade de pçs por afiação Target 1950'. No entanto algumas features apresentaram relações fortíssimas. Relações desse tipo indicam redundancia nas medições e pode ser descartadas para melhorar a eficiência do futuro modelo preditivo. Por este motivo serão descartadas as seguintes colunas:

* 'Vc m/min' : Correlação de 99% com 'Rotação RPM'
* 'CT (s)' : Correlação de 99% com 'Rotação RPM'
* 'Sub-shift mm' :  Correlação de 100% com 'Shifting mm'

In [30]:
df_corrigido

,Revestimento,Avanço mm/rev.,Rotação RPM,Vc m/min,Shifting mm,Sub-Shift mm,CT (S),Quantidade de pçs por afiação Target 1950,dureza sup,dureza nuc
0,Alcrona Pro,1.75,350,98.960,10.0,0.666667,42.9,2620,94.3,96.5
1,Alcrona Evo,1.40,390,110.270,11.4,0.760000,38.5,2458,90.4,93.1
2,Alcrona Pro,1.75,320,90.478,11.4,0.760000,46.9,2718,98.6,94.6
3,Alcrona Evo,1.40,320,90.478,10.0,0.666667,46.9,2496,93.9,94.4
4,Alcrona Evo,2.00,510,144.199,10.0,0.666667,29.4,1872,92.0,93.2
...,...,...,...,...,...,...,...,...,...,...
177,Alcrona Evo,1.75,350,98.960,10.0,0.666667,42.9,2414,95.2,93.7
180,Alcrona Evo,1.40,510,144.199,11.4,0.760000,29.4,2336,97.5,98.2
181,Alcrona Evo,1.50,320,90.478,11.4,0.760000,46.9,2453,92.2,96.2
183,Alcrona Pro,1.40,320,90.478,10.0,0.666667,46.9,3125,94.4,92.2


In [31]:
# seleciona as features que serão usadas no modelo

df_selecionado = df_corrigido # copia o df_corrigido em df_selecionado
df_selecionado = df_selecionado.drop(columns=['Vc m/min', 'CT (S)', 'Sub-Shift mm']) # descarta as colunas 'Vc m/min', 'CT (S)' e 'Sub-Shift mm'
df_selecionado.head() # exibe df_selecionado

,Revestimento,Avanço mm/rev.,Rotação RPM,Shifting mm,Quantidade de pçs por afiação Target 1950,dureza sup,dureza nuc
0,Alcrona Pro,1.75,350,10.0,2620,94.3,96.5
1,Alcrona Evo,1.40,390,11.4,2458,90.4,93.1
2,Alcrona Pro,1.75,320,11.4,2718,98.6,94.6
3,Alcrona Evo,1.40,320,10.0,2496,93.9,94.4
4,Alcrona Evo,2.00,510,10.0,1872,92.0,93.2


# 3. Feature Engineering (Encoding e divisão target-features)

In [32]:
# One Hot Encoding na variável categórica 'Revestimento'
df_modelo = pd.get_dummies(df_selecionado, columns=['Revestimento'], drop_first=True, dtype=int).reset_index(drop=True)

In [33]:
# divisão target-features de df_modelo

X = df_modelo.drop(columns=['Quantidade de pçs por afiação Target 1950']) # separa todas as features em X
y = df_modelo['Quantidade de pçs por afiação Target 1950'] # separa o alvo em y

# 4. Treinamento e avaliação

In [34]:
# importa bibliotecas necessárias para treinamento e avaliação
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

In [35]:
# divide X e y em treino e teste na proporção 80/20
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=42)

In [36]:
# instancia o modelo
modelo_random_forest = RandomForestRegressor(n_estimators=100, random_state=42) # n_estimators=100 significa que o modelo criará 100 "árvores de decisão" diferentes

In [37]:
modelo_random_forest.fit(X_treino, y_treino)

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease o

In [38]:
# realiza a previsão sobre o X_teste
previsao = modelo_random_forest.predict(X_teste)

In [39]:
# calcular as métricas de desempenho

mae = mean_absolute_error(y_teste, previsao) # calcula o Mean Absolute Error
rmse = np.sqrt(mean_squared_error(y_teste, previsao)) # calcula Root Mean Square Error
r2 = r2_score(y_teste, previsao) # calcula R2

print("\n--- Desempenho do Modelo ---")
print(f"MAE (Erro Médio Absoluto): {mae:.2f} peças") # representa, em média, por quantas peças o modelo está errando
print(f"RMSE (Erro Quadrático Médio): {rmse:.2f} peças") # similar ao MAE, mas pune com maior rigor os erros mais discrepantes por se tratar de um cálculo quadrático
print(f"R² Score: {r2:.2f}") # é a previsibilidade do seu modelo. Quanto de y_teste é explicado pelos valores de X_teste


--- Desempenho do Modelo ---
MAE (Erro Médio Absoluto): 157.88 peças
RMSE (Erro Quadrático Médio): 209.49 peças
R² Score: 0.65
